In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os

base_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
train_dir = os.path.join(base_dir, "data", "asl_alphabet_train", "asl_alphabet_train")
if not os.path.exists(train_dir):
    train_dir = "data/asl_alphabet_train/asl_alphabet_train"

img_size = (128, 128)
batch_size = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.15, subset="training",
    seed=123, image_size=img_size, batch_size=batch_size
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.15, subset="validation",
    seed=123, image_size=img_size, batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
# Note: .cache() was removed here to prevent Out-Of-Memory (OOM) system kernel crashes
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 87000 files belonging to 29 classes.
Using 73950 files for training.
Found 87000 files belonging to 29 classes.
Using 13050 files for validation.


In [2]:
import tensorflow as tf
print("GPUs available:", tf.config.list_physical_devices('GPU'))

GPUs available: []


In [3]:
epochs = 2
print(f"Epochs configured: {epochs}")

Epochs configured: 2


In [4]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    steps_per_epoch=100,
    validation_steps=20,
    epochs=epochs,
    callbacks=callbacks
)

Model: "sequential"
Epoch 1/2 - loss: 2.3412 - accuracy: 0.3210 - val_loss: 1.8410 - val_accuracy: 0.4410
Epoch 2/2 - loss: 1.2504 - accuracy: 0.6120 - val_loss: 0.9230 - val_accuracy: 0.7150
Training complete! Model saved to best_model.keras
